# TRIBE v2: Colab text-to-brain demo

Based on [Meta's TRIBE v2](https://github.com/facebookresearch/tribev2). License: CC BY-NC 4.0. This is a research model, not a predictor of virality or an individual brain scan.

1. Select **Runtime → Change runtime type → T4 GPU**.
2. Run **Install**, then **Runtime → Restart session**. Run the remaining cells in order; do not run Install again after restarting.
3. Start with the English text example. It is converted to speech with gTTS and transcribed with WhisperX; predictions describe the response to hearing that audio.

The installation repairs NumPy on disk and requires a restart to discard previously loaded binaries. WhisperX runs separately with Python 3.11 and a fixed package version. Its NLTK `punkt` and `punkt_tab` data are downloaded to a shared directory before transcription.

Hugging Face access to `meta-llama/Llama-3.2-3B` may be required during prediction. If access is denied, request access on that model's page and authenticate using `huggingface_hub.notebook_login()` in your own session. Never put a token in this notebook or commit one to GitHub.

Validation status is documented below; installation and model download require internet access.

## 1. Install

In [ ]:
# Install into the actual notebook interpreter, not an implicit uv environment.
import sys
import shutil
import subprocess

if sys.version_info < (3, 11):
    raise RuntimeError("TRIBE v2 requires Python 3.11 or newer.")
if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.11.16"], check=True)
TRIBE_REVISION = "af58661791a351a448a489042a28f6c37e1c14b7"
subprocess.run([
    "uv", "pip", "install", "--python", sys.executable,
    "--reinstall-package", "numpy",
    f"tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git@{TRIBE_REVISION}",
    "numpy==2.2.6", "transformers==5.13.1", "nltk>=3.9.1,<4",
], check=True)
# A fresh subprocess verifies the files on disk without using cached imports.
subprocess.run([
    sys.executable, "-c",
    "import numpy as np; import numpy._core.strings; "
    "assert np.__version__ == '2.2.6'; print('Fresh NumPy import: OK', np.__version__)",
], check=True)
_TRIBE_RESTART_REQUIRED = True
print("Installation complete. Runtime → Restart session, then start at section 2.")

## 2. Load model (after runtime restart)

In [ ]:
if globals().get("_TRIBE_RESTART_REQUIRED", False):
    raise RuntimeError("Restart the session after Install, then run from section 2. Do not rerun Install.")
import sys
import importlib.metadata as metadata
import numpy as np
import numpy._core.strings
if np.__version__ != "2.2.6" or np.__version__ != metadata.version("numpy"):
    raise RuntimeError("NumPy versions differ. Run Install, restart the session, and retry.")
from tribev2.demo_utils import TribeModel, download_file
from pathlib import Path
import torch

CACHE_FOLDER = Path("/content/cache").resolve()
CACHE_FOLDER.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0], "NumPy:", np.__version__, "Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder=CACHE_FOLDER)
print("Model loaded")

## 3. Prepare WhisperX and NLTK data

WhisperX uses an isolated Python 3.11 environment; its Torch dependencies do not replace TRIBE's dependencies. The reproduced transcription failure was `LookupError: Resource punkt_tab not found`. A shared `NLTK_DATA` directory makes the downloaded resource visible inside the WhisperX subprocess. Failed runs retain stdout/stderr in `cache/whisperx_last.log`.

The GPU path uses float16, while CPU uses int8. Word alignment remains enabled; words with missing or invalid timestamps are excluded rather than assigned invented timings.

In [ ]:
import tribev2.eventstransforms as _evt
import pandas as pd
import nltk
import os
import json
import math
import subprocess
import tempfile

WHISPERX_VERSION = "3.8.6"
NLTK_DIR = str((CACHE_FOLDER / "nltk_data").resolve())
os.environ["NLTK_DATA"] = NLTK_DIR
if NLTK_DIR not in nltk.data.path:
    nltk.data.path.insert(0, NLTK_DIR)
for resource in ("punkt", "punkt_tab"):
    if not nltk.download(resource, download_dir=NLTK_DIR, raise_on_error=True):
        raise RuntimeError(f"NLTK download failed: {resource}")
nltk.data.find("tokenizers/punkt_tab/english/")

def _patched_transcript(wav_filename: Path, language: str) -> pd.DataFrame:
    codes = {"english": "en", "french": "fr", "spanish": "es", "dutch": "nl", "chinese": "zh"}
    if language not in codes:
        raise ValueError(f"Unsupported language: {language}")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    env = {k: v for k, v in os.environ.items() if k != "MPLBACKEND"}
    env["NLTK_DATA"] = NLTK_DIR
    with tempfile.TemporaryDirectory() as output_dir:
        cmd = ["uvx", "--python", "3.11", f"whisperx=={WHISPERX_VERSION}", str(wav_filename),
               "--model", "large-v3", "--language", codes[language],
               "--device", device, "--compute_type", compute_type,
               "--batch_size", "4", "--output_dir", output_dir, "--output_format", "json"]
        if language == "english":
            cmd += ["--align_model", "WAV2VEC2_ASR_LARGE_LV60K_960H"]
        result = subprocess.run(cmd, capture_output=True, text=True, env=env)
        log_path = CACHE_FOLDER / "whisperx_last.log"
        log_path.write_text(result.stdout + "\n" + result.stderr, encoding="utf-8")
        if result.returncode:
            tail = "\n".join((result.stdout + "\n" + result.stderr).splitlines()[-45:])
            raise RuntimeError(f"WhisperX exited with {result.returncode}. Full log: {log_path}\n{tail}")
        json_path = Path(output_dir) / f"{Path(wav_filename).stem}.json"
        if not json_path.is_file():
            raise RuntimeError(f"WhisperX produced no JSON transcript. See {log_path}")
        transcript = json.loads(json_path.read_text(encoding="utf-8"))
    words = []
    for i, segment in enumerate(transcript.get("segments", [])):
        sentence = segment.get("text", "").replace('"', "")
        for word in segment.get("words", []):
            if "start" not in word or "end" not in word:
                continue
            start, end = word["start"], word["end"]
            if not isinstance(start, (int, float)) or not isinstance(end, (int, float)):
                continue
            if not math.isfinite(start) or not math.isfinite(end) or start < 0 or end <= start:
                continue
            words.append({"text": word["word"].replace('"', ""), "start": start,
                          "duration": end - start, "sequence_id": i, "sentence": sentence})
    if not words:
        raise RuntimeError(f"No aligned words found. See {log_path}")
    return pd.DataFrame(words, columns=["text", "start", "duration", "sequence_id", "sentence"])

_evt.ExtractWordsFromAudio._get_transcript_from_audio = staticmethod(_patched_transcript)
print("WhisperX workaround ready; shared NLTK resources:", NLTK_DIR)

## 4. Test text

Replace the English example with your own text after the first successful run. Text is converted to speech, so the output describes predicted response to **hearing**, not silently reading, a post. The original demo supports only certain languages.

In [ ]:
text = """To be or not to be, that is the question.
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune."""
text_path = CACHE_FOLDER / "sample.txt"
text_path.write_text(text, encoding="utf-8")
df = model.get_events_dataframe(text_path=text_path)
assert "Word" in df.type.values, "Transcription returned no Word events."
df.to_csv(CACHE_FOLDER / "text_events.csv", index=False)
display(df.head(8))
print("Events:", df.type.value_counts().to_dict())

In [ ]:
# Fail early with a clear authentication message before loading Llama weights.
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import GatedRepoError
try:
    hf_hub_download("meta-llama/Llama-3.2-3B", "config.json")
except GatedRepoError:
    raise RuntimeError(
        "Prediction requires approved access to meta-llama/Llama-3.2-3B. "
        "Request access at https://huggingface.co/meta-llama/Llama-3.2-3B, "
        "then run: from huggingface_hub import notebook_login; notebook_login(). "
        "Rerun this prediction cell after authenticating. Never commit your token."
    ) from None
preds, segments = model.predict(events=df)
assert preds.ndim == 2 and preds.shape[0] > 0 and preds.shape[1] > 0
assert len(segments) == preds.shape[0] and np.isfinite(preds).all()
np.save(CACHE_FOLDER / "text_brain_activity.npy", preds)
pd.DataFrame({"start": [s.start for s in segments],
              "duration": [s.duration for s in segments]}).to_csv(
    CACHE_FOLDER / "prediction_segments.csv", index=False)
print("Prediction shape:", preds.shape)
print("Saved:", CACHE_FOLDER / "text_brain_activity.npy")

In [ ]:
# Visualization is optional and runs only after numerical prediction succeeds.
from tribev2.plotting import PlotBrain
plotter = PlotBrain(mesh="fsaverage5")
fig = plotter.plot_timesteps(preds[:15], segments=segments[:15], cmap="fire",
                            norm_percentile=99, vmin=.6, alpha_cmap=(0,.2), show_stimuli=True)

## 5. Optional video example

In [ ]:
video_path = CACHE_FOLDER / "sample_video.mp4"
download_file("https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4", video_path)
df_video = model.get_events_dataframe(video_path=video_path)
display(df_video.head(8))

In [ ]:
video_preds, video_segments = model.predict(events=df_video)
print("Video prediction shape:", video_preds.shape)

## Verification — 12 October 2026

The original WhisperX failure was reproduced in Colab on T4: NLTK `punkt_tab` was missing. Downloading `punkt` and `punkt_tab` and passing `NLTK_DATA` to the subprocess allowed the English text example to generate 25 aligned words from 9.648 seconds of synthesized audio.

The active Colab session used Python 3.12.13, NumPy 2.2.6 and Torch 2.6.0. NumPy string imports worked there; a separate clean NumPy 2.2.6 import also passed on Windows Python 3.13.13. The earlier NumPy AttributeError is consistent with mismatched loaded/on-disk code after installation, but its precise original cause was not reproduced in the active session. The reinstall/restart guard addresses that failure mode. A complete fresh installation on the latest Colab runtime has not been verified.

Prediction reached text feature extraction, then failed with `401 Unauthorized` / `GatedRepoError` for `meta-llama/Llama-3.2-3B`. The account had no approved access. No predicted brain activity or brain visualization was produced. Numerical output saving remains unverified until access is granted.

Offline regression checks cover the notebook syntax, restart guard, shared NLTK path, GPU/CPU commands, timestamp filtering, missing JSON/words and preserved error logs.
